# Daily Summary ETL (Group 02 · Step 01) — SYNTHETIC DEMO

> **What this notebook does** — the same `02-01` ETL as the real pipeline, but run on the **synthetic** `dsptlp/synthetic-market-data` (+ shard 2) dataset instead of the licensed S3 bucket. It collapses the synthetic minute bars into per-day first/last-minute price summaries and daily volume, writing them **locally** to `/kaggle/working` (no S3 egress, no licensed data).

## Job

| | |
|---|---|
| **Reads (input)** | Kaggle `dsptlp/synthetic-market-data` + `dsptlp/synthetic-market-data-02` → `minute_data_final/*.parquet` (20,000 synthetic tickers) |
| **Writes (output)** | `/kaggle/working/summary/minute_summary/data.parquet` + `/kaggle/working/summary/daily_volume/data.parquet` |
| **Libraries** | `pyspark` (local) |
| **Pipeline role** | stage 02 — aggregated summary feeding `03-01` (demo) |

Run `demo/03-01-correlation.ipynb` after this one to see the lead-lag analysis on the synthetic universe.


In [ ]:
# ============================================================================
# Setup -- secrets, S3 client, DuckDB S3 helper, Spark session
# ============================================================================

!pip install -q duckdb --upgrade

import sys
import os
import shutil
import json

# Copy the autotrade package files from the Kaggle dataset into a proper
# package directory, then add it to sys.path.
# Locate the autotrade-package dataset wherever it mounts (mount path varies).
import glob as _glob
_pkg_cands = sorted(_glob.glob("/kaggle/input/**/creds.json", recursive=True))
if _pkg_cands:
    src_pkg = os.path.dirname(_pkg_cands[0])
else:
    src_pkg = '/kaggle/input/autotrade-package'
dst_pkg = '/kaggle/working/autotrade'
if os.path.isdir(src_pkg) and not os.path.isdir(dst_pkg):
    os.makedirs(dst_pkg, exist_ok=True)
    for fname in os.listdir(src_pkg):
        if fname.endswith('.py'):
            shutil.copy2(os.path.join(src_pkg, fname), os.path.join(dst_pkg, fname))
    print(f"Copied autotrade package from dataset to working directory")

sys.path.insert(0, '/kaggle/working')

import time
from io import BytesIO

import numpy as np
import pandas as pd
import boto3
import duckdb
import math
from datetime import datetime, timedelta

from IPython.display import display, HTML

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import *

from autotrade.config import load_config
from autotrade.storage import s3_client, duckdb_s3_connect as _duckdb_s3_connect, spark_session

# Load credentials from autotrade-package dataset (instead of UserSecretsClient)
with open(os.path.join(src_pkg, 'creds.json')) as f:
    creds = json.load(f)

cfg = load_config()
cfg.aws_access_key_id = creds["aws_access_key_id"]
cfg.aws_secret_access_key = creds["aws_secret_access_key"]
cfg.massive_api_key = creds["massive_api_key"]

REGION_NAME = cfg.aws_region
MY_BUCKET   = cfg.s3_bucket

region_name = REGION_NAME
my_bucket   = MY_BUCKET

SPARK_TMP   = cfg.spark_tmp
os.makedirs(SPARK_TMP, exist_ok=True)

s3 = s3_client(cfg)


def duckdb_s3_connect():
    return _duckdb_s3_connect(cfg)


spark = spark_session(cfg)

print("Setup complete")


In [ ]:
# Explicit schema -- prices/volume double, symbol/trades long. `volume` is double
# to absorb the int64/double drift across the per-ticker files.
MINUTE_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("date",   LongType(),   True),
    StructField("open",   DoubleType(), True),
    StructField("high",   DoubleType(), True),
    StructField("low",    DoubleType(), True),
    StructField("close",  DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("vwap",   DoubleType(), True),
    StructField("trades", LongType(),   True),
])

# Synthetic minute bars live in the mounted datasets (mount path varies).
# A single Kaggle session cannot safely read ALL 20,000 files (~83 GB), so we
# sample a deterministic subset for the demo. Bump N_TICKERS to process more.
import glob as _glob
import random as _random

N_TICKERS = 3000

minute_dirs = sorted(_glob.glob("/kaggle/input/**/minute_data_final", recursive=True))
if not minute_dirs:
    raise SystemExit("synthetic minute_data_final not found in /kaggle/input - attach "
                     "dsptlp/synthetic-market-data and dsptlp/synthetic-market-data-02")
print("Synthetic minute dirs:", minute_dirs)

all_min_files = []
for d in minute_dirs:
    all_min_files += [os.path.join(d, f) for f in sorted(os.listdir(d)) if f.endswith(".parquet")]

# Per-sector sample that ALWAYS includes the sector leader plus followers:
# sector s holds indices s, s+200, s+400, ... (leader = index s). Taking the
# first N_SECTORS_PER_SAMPLE indices of each sector guarantees aligned
# (leader, follower) pairs exist for the correlation stage.
N_SECTORS = 200
PER_SECTOR = max(1, N_TICKERS // N_SECTORS)      # 15
sample_files = []
by_sector = {}
for f in all_min_files:
    idx = int(os.path.basename(f)[3:8])          # "SGT00003.parquet" -> 3
    by_sector.setdefault(idx % N_SECTORS, []).append((idx, f))
for s in range(N_SECTORS):
    members = sorted(by_sector.get(s, []))       # sorted by global index
    sample_files += [f for idx, f in members[:PER_SECTOR]]
print(f"Reading {len(sample_files):,} synthetic ticker files "
      f"({PER_SECTOR} per sector incl. leaders, of {len(all_min_files):,} available)")

df_raw = spark.read.schema(MINUTE_SCHEMA).parquet(*sample_files)

df = df_raw

# Derive the trading day from the epoch-ms timestamp; drop volume (raw-only column).
df = (
    df
    .withColumn("trade_date_time", F.to_timestamp(F.col("date") / 1000))
    .withColumn("trade_date", F.date_trunc("day", F.col("trade_date_time")))
    .drop("volume")
)

# Rank rows within each (symbol, day) ascending and descending.
window_asc  = Window.partitionBy("symbol", "trade_date").orderBy("trade_date_time")
window_desc = Window.partitionBy("symbol", "trade_date").orderBy(F.col("trade_date_time").desc())

df = (
    df
    .withColumn("rn_asc",  F.row_number().over(window_asc))
    .withColumn("rn_desc", F.row_number().over(window_desc))
)



In [ ]:
# ============================================================================
# Daily volume & trades summary (all tickers, per trading day)
# ============================================================================

df_daily_vol = (
    df_raw
    .withColumn("trade_date", F.date_trunc("day", F.to_timestamp(F.col("date") / 1000)))
    .groupBy("symbol", "trade_date")
    .agg(
        F.sum("volume").alias("volume"),
        F.sum("trades").alias("trades"),
        F.last("close").alias("close"),
    )
)

(
    df_daily_vol.write
    .mode("overwrite")
    .parquet("/kaggle/working/summary/daily_volume/data.parquet")
)

print("Wrote /kaggle/working/summary/daily_volume/data.parquet")


In [ ]:
# Keep the first and last minute bar of each (symbol, day).
df_filtered = df.filter((F.col("rn_asc") == 1) | (F.col("rn_desc") == 1))

(
    df_filtered.write
    .mode("overwrite")
    .parquet("/kaggle/working/summary/minute_summary/data.parquet")
)

print("Wrote /kaggle/working/summary/minute_summary/data.parquet")

spark.stop()



In [ ]:
# ============================================================================
# Publish the derived summaries so demo/03-01 can read them in another session
# (mirrors the real pipeline writing summaries to S3, then reading them).
# ============================================================================
import json as _json
import shutil as _sh

UPLOAD_SUM = "/kaggle/working/upload_summary"
os.makedirs(f"{UPLOAD_SUM}/minute_summary", exist_ok=True)
os.makedirs(f"{UPLOAD_SUM}/daily_volume", exist_ok=True)
if os.path.isdir("/kaggle/working/summary/minute_summary/data.parquet"):
    _sh.copytree("/kaggle/working/summary/minute_summary/data.parquet",
                 f"{UPLOAD_SUM}/minute_summary/data.parquet", dirs_exist_ok=True)
if os.path.isdir("/kaggle/working/summary/daily_volume/data.parquet"):
    _sh.copytree("/kaggle/working/summary/daily_volume/data.parquet",
                 f"{UPLOAD_SUM}/daily_volume/data.parquet", dirs_exist_ok=True)

_meta = {
    "title": "Synthetic Market Data Summaries",
    "id": "dsptlp/synthetic-market-data-summary",
    "isPrivate": False,
    "licenses": [{"name": "CC0-1.0"}],
}
with open(f"{UPLOAD_SUM}/dataset-metadata.json", "w") as f:
    _json.dump(_meta, f, indent=2)

# --- Kaggle auth from env vars (Add-ons -> Secrets on Kaggle) ------------
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "dsptlp")
KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN", "")

os.makedirs("/root/.kaggle", exist_ok=True)
if KAGGLE_API_TOKEN:
    with open("/root/.kaggle/access_token", "w") as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod("/root/.kaggle/access_token", 0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
!pip install -q --upgrade kaggle

rc = os.system(
    f"kaggle datasets version -p {UPLOAD_SUM} --dir-mode zip "
    f"-m \"synthetic summaries (demo {len(sample_files):,} tickers)\""
)
print("Publish rc =", rc)
if rc != 0:
    print("Publish failed - the summaries remain in /kaggle/working/summary for same-session use")
